In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)

In [2]:
# =========================
# 0. Config
# =========================

PROJECT_ROOT = Path("/home/zilinm2/futures_anomaly_project")

NB01_DIR = PROJECT_ROOT / "notebook01_base_clean_panel"
NB02_DIR = PROJECT_ROOT / "notebook02_vol_sorted_technical_anomaly"

NB03_DIR = PROJECT_ROOT / "notebook03_liquidity_attenuation"
FEATURE_DIR = NB03_DIR / "features"
STRATEGY_DIR = NB03_DIR / "strategy_returns"
TABLE_DIR = NB03_DIR / "tables"
DIAG_DIR = NB03_DIR / "diagnostics"

for d in [NB03_DIR, FEATURE_DIR, STRATEGY_DIR, TABLE_DIR, DIAG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLEAN_PANEL_PARQUET = NB01_DIR / "notebook01_base_clean_panel.parquet"
CLEAN_PANEL_CSV = NB01_DIR / "notebook01_base_clean_panel.csv"

NB02_STRATEGY_PARQUET = NB02_DIR / "strategy_returns" / "notebook02_symbol_ma_strategy_returns.parquet"
NB02_STRATEGY_CSV = NB02_DIR / "strategy_returns" / "notebook02_symbol_ma_strategy_returns.csv"

ACTIVITY_WINDOW = 252
MIN_ACTIVITY_OBS = 60
TRADING_DAYS_PER_YEAR = 252

ACTIVITY_BUCKET_ORDER = ["Low", "Mid", "High"]
VOL_BUCKET_ORDER = ["Low", "Mid", "High"]

RETURN_COLS = [
    "gross_strategy_return",
    "net_strategy_return",
    "passive_return",
    "gross_map_vs_passive",
    "net_map_vs_passive",
    "net_map_vs_zero",
]

CORE_RETURN_COLS = [
    "net_strategy_return",
    "net_map_vs_passive",
    "net_map_vs_zero",
]

RUN_CONFIG = {
    "notebook": "notebook03_liquidity_attenuation",
    "version": "v2_lagged_activity_audited",
    "input_clean_panel_parquet": str(CLEAN_PANEL_PARQUET),
    "input_clean_panel_csv": str(CLEAN_PANEL_CSV),
    "input_notebook02_strategy_parquet": str(NB02_STRATEGY_PARQUET),
    "input_notebook02_strategy_csv": str(NB02_STRATEGY_CSV),
    "activity_window": ACTIVITY_WINDOW,
    "min_activity_obs": MIN_ACTIVITY_OBS,
    "activity_bucket_method": "within_symbol_rolling_percentile",
    "main_activity_timing": "lagged_signal_date_activity_state",
    "strategy_timing": "Notebook02 delayed execution; exec_position[t] uses signal[t-1]",
    "return_timing": "strategy return at date t is close[t] to close[t+1]",
    "activity_data_availability_assumption": "signal-date volume, open interest, and turnover are known after signal-date close and before next-trading-day execution",
    "no_lookahead_rule": "activity_bucket used for analysis is computed at signal_date < execution date",
    "annualization": TRADING_DAYS_PER_YEAR,
}

print(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))


{
  "notebook": "notebook03_liquidity_attenuation",
  "version": "v2_lagged_activity_audited",
  "input_clean_panel_parquet": "/home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.parquet",
  "input_clean_panel_csv": "/home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.csv",
  "input_notebook02_strategy_parquet": "/home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/strategy_returns/notebook02_symbol_ma_strategy_returns.parquet",
  "input_notebook02_strategy_csv": "/home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/strategy_returns/notebook02_symbol_ma_strategy_returns.csv",
  "activity_window": 252,
  "min_activity_obs": 60,
  "activity_bucket_method": "within_symbol_rolling_percentile",
  "main_activity_timing": "lagged_signal_date_activity_state",
  "strategy_timing": "Notebook02 delayed execution; exec_position[t] uses signal[t-1]",
  "return_timing

In [3]:
# =========================
# 1. Helper functions
# =========================

def read_table(parquet_path: Path, csv_path: Path, date_cols=None) -> pd.DataFrame:
    date_cols = date_cols or []
    if parquet_path.exists():
        print(f"Loading parquet: {parquet_path}")
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        print(f"Loading csv: {csv_path}")
        return pd.read_csv(csv_path, parse_dates=date_cols, low_memory=False)
    raise FileNotFoundError(
        "No input table found. Checked:\n"
        f"{parquet_path}\n{csv_path}"
    )


def safe_to_parquet(df: pd.DataFrame, path: Path) -> bool:
    try:
        df.to_parquet(path, index=False)
        return True
    except Exception as exc:
        print(f"[WARN] parquet save failed: {path}")
        print(f"[WARN] {type(exc).__name__}: {exc}")
        return False


def save_table(df: pd.DataFrame, out_dir: Path, name: str) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / f"{name}.csv"
    parquet_path = out_dir / f"{name}.parquet"
    df.to_csv(csv_path, index=False)
    parquet_ok = safe_to_parquet(df, parquet_path)
    return {
        "name": name,
        "rows": int(len(df)),
        "cols": int(df.shape[1]),
        "csv": str(csv_path),
        "parquet": str(parquet_path) if parquet_ok else None,
    }


def require_columns(df: pd.DataFrame, required_cols, table_name: str):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{table_name} missing required columns: {missing}")


def parse_bool_flag(s: pd.Series, name: str) -> pd.Series:
    """
    Convert input flag columns to strict booleans.

    This avoids pandas treating non-empty strings such as "False" or "0" as True.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)

    if pd.api.types.is_numeric_dtype(s):
        valid = s.dropna()
        bad = ~valid.isin([0, 1])
        if bad.any():
            raise ValueError(
                f"Invalid numeric boolean flag values in {name}: "
                f"{sorted(valid[bad].unique())[:10]}"
            )
        return s.fillna(0).astype(int).astype(bool)

    normalized = s.astype("string").str.strip().str.lower()
    normalized = normalized.mask(normalized == "")
    mapped = normalized.map({
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
        "y": True, "n": False,
    })
    bad = mapped.isna() & normalized.notna()
    if bad.any():
        raise ValueError(
            f"Invalid boolean flag values in {name}: "
            f"{sorted(s[bad].dropna().astype(str).unique())[:10]}"
        )
    return mapped.fillna(False).astype(bool)


def annualized_perf(x: pd.Series, trading_days: int = 252) -> dict:
    x = pd.to_numeric(x, errors="coerce").dropna()
    n = int(len(x))
    if n == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "mean_ann": np.nan,
            "vol_daily": np.nan,
            "vol_ann": np.nan,
            "sharpe": np.nan,
            "t_stat": np.nan,
            "min_daily": np.nan,
            "max_daily": np.nan,
            "max_drawdown": np.nan,
        }

    mean_daily = float(x.mean())
    vol_daily = float(x.std(ddof=1)) if n > 1 else np.nan
    mean_ann = mean_daily * trading_days
    vol_ann = vol_daily * math.sqrt(trading_days) if pd.notna(vol_daily) else np.nan
    sharpe = mean_ann / vol_ann if vol_ann and vol_ann > 0 else np.nan
    t_stat = mean_daily / (vol_daily / math.sqrt(n)) if vol_daily and vol_daily > 0 else np.nan

    equity = (1.0 + x).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    max_dd = float(drawdown.min()) if len(drawdown) else np.nan

    return {
        "n_obs": n,
        "mean_daily": mean_daily,
        "mean_ann": float(mean_ann),
        "vol_daily": vol_daily,
        "vol_ann": vol_ann,
        "sharpe": float(sharpe) if pd.notna(sharpe) else np.nan,
        "t_stat": float(t_stat) if pd.notna(t_stat) else np.nan,
        "min_daily": float(x.min()),
        "max_daily": float(x.max()),
        "max_drawdown": max_dd,
    }


def summarize_returns(df: pd.DataFrame, group_cols, return_cols) -> pd.DataFrame:
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False, observed=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        base = dict(zip(group_cols, keys))
        for rc in return_cols:
            stats = annualized_perf(g[rc], TRADING_DAYS_PER_YEAR)
            rows.append({**base, "return_col": rc, **stats})
    return pd.DataFrame(rows)


def rolling_zscore_by_symbol(df: pd.DataFrame, value_col: str) -> pd.Series:
    g = df.groupby("symbol", sort=False)[value_col]
    mean = g.transform(lambda s: s.rolling(ACTIVITY_WINDOW, min_periods=MIN_ACTIVITY_OBS).mean())
    std = g.transform(lambda s: s.rolling(ACTIVITY_WINDOW, min_periods=MIN_ACTIVITY_OBS).std(ddof=1))
    z = (df[value_col] - mean) / std.replace(0, np.nan)
    return z.clip(-5, 5)


def last_rank_pct_raw(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    if len(x) < MIN_ACTIVITY_OBS:
        return np.nan
    last = x[-1]
    less = np.sum(x < last)
    equal = np.sum(x == last)
    return float((less + 0.5 * equal) / len(x))


def assign_activity_bucket(pct: pd.Series) -> pd.Series:
    out = pd.Series(pd.NA, index=pct.index, dtype="object")
    out[pct <= 1.0 / 3.0] = "Low"
    out[(pct > 1.0 / 3.0) & (pct < 2.0 / 3.0)] = "Mid"
    out[pct >= 2.0 / 3.0] = "High"
    return out


def ols_hc1(y: np.ndarray, X: np.ndarray, names) -> pd.DataFrame:
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)

    mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
    y = y[mask]
    X = X[mask]
    n, k = X.shape

    if n <= k + 5:
        return pd.DataFrame({
            "term": names,
            "coef": np.nan,
            "std_err_hc1": np.nan,
            "t_stat_hc1": np.nan,
            "n_obs": n,
            "r2": np.nan,
        })

    xtx_inv = np.linalg.pinv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    resid = y - X @ beta

    meat = X.T @ ((resid ** 2)[:, None] * X)
    cov = (n / max(n - k, 1)) * xtx_inv @ meat @ xtx_inv
    se = np.sqrt(np.maximum(np.diag(cov), 0.0))
    t = beta / se

    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return pd.DataFrame({
        "term": names,
        "coef": beta,
        "std_err_hc1": se,
        "t_stat_hc1": t,
        "n_obs": n,
        "r2": r2,
    })


def activity_effect_flag(diff_t: float, diff_mean_ann: float, n_obs: int) -> str:
    if pd.isna(diff_t) or pd.isna(diff_mean_ann) or n_obs < 100:
        return "INSUFFICIENT_SAMPLE"
    if diff_t <= -1.96 and diff_mean_ann < 0:
        return "ATTENUATED"
    if diff_t >= 1.96 and diff_mean_ann > 0:
        return "ENHANCED_BY_ACTIVITY"
    if diff_mean_ann < 0:
        return "WEAK_ATTENUATION"
    if diff_mean_ann > 0:
        return "WEAK_ENHANCEMENT"
    return "NO_ACTIVITY_EFFECT"


def regression_effect_flag(t, coef, n):
    if pd.isna(t) or pd.isna(coef) or n < 100:
        return "INSUFFICIENT_SAMPLE"
    if t <= -1.96 and coef < 0:
        return "ATTENUATED"
    if t >= 1.96 and coef > 0:
        return "ENHANCED_BY_ACTIVITY"
    if coef < 0:
        return "WEAK_ATTENUATION"
    if coef > 0:
        return "WEAK_ENHANCEMENT"
    return "NO_ACTIVITY_EFFECT"


print("Helper functions loaded.")


Helper functions loaded.


In [4]:
# =========================
# 2. Load inputs
# =========================

clean = read_table(
    CLEAN_PANEL_PARQUET,
    CLEAN_PANEL_CSV,
    date_cols=["date", "trading_day", "datetime"],
)

strategy = read_table(
    NB02_STRATEGY_PARQUET,
    NB02_STRATEGY_CSV,
    date_cols=["date", "signal_date"],
)

for c in ["date", "trading_day", "datetime"]:
    if c in clean.columns:
        clean[c] = pd.to_datetime(clean[c], errors="coerce")

for c in ["date", "signal_date"]:
    if c in strategy.columns:
        strategy[c] = pd.to_datetime(strategy[c], errors="coerce")

clean = clean.sort_values(["symbol", "date"]).reset_index(drop=True)
strategy = strategy.sort_values(["symbol", "ma_window", "version", "date"]).reset_index(drop=True)

# 02 output may use concise names.
if "exec_position" not in strategy.columns and "position" in strategy.columns:
    strategy["exec_position"] = strategy["position"]

if "exec_vol_bucket" not in strategy.columns and "vol_bucket" in strategy.columns:
    strategy["exec_vol_bucket"] = strategy["vol_bucket"]

clean_required = [
    "date",
    "symbol",
    "volume",
    "open_interest",
    "turnover_value",
    "log_volume",
    "log_open_interest",
    "log_turnover_value",
    "tradable_row_flag",
]

strategy_required = [
    "date",
    "signal_date",
    "symbol",
    "ma_window",
    "version",
    "exec_position",
    "exec_vol_bucket",
    "gross_strategy_return",
    "net_strategy_return",
    "passive_return",
    "gross_map_vs_passive",
    "net_map_vs_passive",
    "net_map_vs_zero",
]

require_columns(clean, clean_required, "Notebook01 clean panel")
require_columns(strategy, strategy_required, "Notebook02 strategy returns")

# Normalize flags once before any filtering. Avoid pandas string-to-bool pitfalls.
clean["tradable_row_flag"] = parse_bool_flag(clean["tradable_row_flag"], "tradable_row_flag")
assert pd.api.types.is_bool_dtype(clean["tradable_row_flag"]), "tradable_row_flag was not converted to bool."

available_return_cols = [c for c in RETURN_COLS if c in strategy.columns]
available_core_return_cols = [c for c in CORE_RETURN_COLS if c in strategy.columns]

print("clean shape:", clean.shape)
print("strategy shape:", strategy.shape)
print("available_return_cols:", available_return_cols)
print("strategy date range:", strategy["date"].min(), "->", strategy["date"].max())
display(strategy.head())


Loading parquet: /home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.parquet
Loading parquet: /home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/strategy_returns/notebook02_symbol_ma_strategy_returns.parquet
clean shape: (58591, 67)
strategy shape: (452150, 26)
available_return_cols: ['gross_strategy_return', 'net_strategy_return', 'passive_return', 'gross_map_vs_passive', 'net_map_vs_passive', 'net_map_vs_zero']
strategy date range: 2016-04-07 00:00:00 -> 2026-06-17 00:00:00


,date,signal_date,symbol,sector,close,fwd_ret_1d,fwd_log_ret_1d,fwd_tradable_return_flag,vol_bucket,rv_60,ma_value,signal_close,ma_window,version,position,valid_strategy_row,turnover,cost_return,passive_return,gross_strategy_return,net_strategy_return,gross_map_vs_passive,net_map_vs_passive,net_map_vs_zero,exec_position,exec_vol_bucket
0,2016-04-07,2016-04-06,AG,precious_metal,3377.0,0.000888,0.000888,True,Low,0.010202,3407.9,3371.0,10,long_flat,0.0,True,0.0,0.0000,0.000888,0.000000,0.000000,-0.000888,-0.000888,0.000000,0.0,Low
1,2016-04-08,2016-04-07,AG,precious_metal,3380.0,0.011243,0.011180,True,Low,0.010204,3398.7,3377.0,10,long_flat,0.0,True,0.0,0.0000,0.011243,0.000000,0.000000,-0.011243,-0.011243,0.000000,0.0,Low
2,2016-04-11,2016-04-08,AG,precious_metal,3418.0,0.014336,0.014234,True,Low,0.010069,3395.8,3380.0,10,long_flat,0.0,True,0.0,0.0000,0.014336,0.000000,0.000000,-0.014336,-0.014336,0.000000,0.0,Low
3,2016-04-12,2016-04-11,AG,precious_metal,3467.0,0.004615,0.004604,True,Mid,0.010170,3396.6,3418.0,10,long_flat,1.0,True,1.0,0.0005,0.004615,0.004615,0.004115,0.000000,-0.000500,0.004115,1.0,Mid
4,2016-04-13,2016-04-12,AG,precious_metal,3483.0,0.000574,0.000574,True,Mid,0.010323,3402.7,3467.0,10,long_flat,1.0,True,0.0,0.0000,0.000574,0.000574,0.000574,0.000000,0.000000,0.000574,1.0,Mid


In [5]:
# =========================
# 3. Build symbol-level activity features
# =========================

activity = clean[clean_required].copy()
activity = activity[activity["tradable_row_flag"]].copy()
activity = activity.sort_values(["symbol", "date"]).reset_index(drop=True)

for col in ["volume", "open_interest", "turnover_value"]:
    if (activity[col] <= 0).any():
        raise ValueError(f"{col} contains non-positive values after tradable filter.")

component_map = {
    "log_volume": "volume_z",
    "log_open_interest": "open_interest_z",
    "log_turnover_value": "turnover_value_z",
}

for src, dst in component_map.items():
    activity[dst] = rolling_zscore_by_symbol(activity, src)

activity["activity_score"] = activity[
    ["volume_z", "open_interest_z", "turnover_value_z"]
].mean(axis=1, skipna=False)

activity["activity_pctile"] = (
    activity.groupby("symbol", sort=False)["activity_score"]
    .transform(
        lambda s: s.rolling(ACTIVITY_WINDOW, min_periods=MIN_ACTIVITY_OBS)
        .apply(last_rank_pct_raw, raw=True)
    )
)

activity["activity_bucket"] = assign_activity_bucket(activity["activity_pctile"])

activity_features = activity[
    [
        "date",
        "symbol",
        "volume_z",
        "open_interest_z",
        "turnover_value_z",
        "activity_score",
        "activity_pctile",
        "activity_bucket",
    ]
].copy()

activity_feature_summary = (
    activity_features.groupby("symbol")
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        obs_count=("date", "size"),
        valid_activity_count=("activity_bucket", lambda x: x.notna().sum()),
        low_activity_count=("activity_bucket", lambda x: (x == "Low").sum()),
        mid_activity_count=("activity_bucket", lambda x: (x == "Mid").sum()),
        high_activity_count=("activity_bucket", lambda x: (x == "High").sum()),
        mean_activity_score=("activity_score", "mean"),
        std_activity_score=("activity_score", "std"),
    )
    .reset_index()
)

print("activity_features shape:", activity_features.shape)
print("valid activity rows:", int(activity_features["activity_bucket"].notna().sum()))
display(activity_feature_summary.head(10))


activity_features shape: (58591, 8)
valid activity rows: 55405


,symbol,start_date,end_date,obs_count,valid_activity_count,low_activity_count,mid_activity_count,high_activity_count,mean_activity_score,std_activity_score
0,AG,2016-01-05,2026-06-18,2538,2420,882,744,794,0.058428,0.961313
1,AL,2016-01-05,2026-06-18,2538,2420,859,716,845,0.119990,1.026388
2,AO,2023-06-19,2026-06-18,726,608,210,191,207,0.439039,0.869672
3,AU,2016-01-05,2026-06-18,2538,2420,835,754,831,0.064914,0.929425
4,CF,2016-01-05,2026-06-18,2538,2420,909,654,857,0.078610,0.993616
5,CU,2016-01-05,2026-06-18,2538,2420,822,760,838,0.009037,0.931057
6,FG,2016-01-05,2026-06-18,2538,2420,893,775,752,0.187690,1.057157
7,HC,2016-01-05,2026-06-18,2538,2420,961,646,813,0.228118,0.948696
8,I,2016-01-05,2026-06-18,2538,2420,853,690,877,-0.146562,1.020779
9,JM,2016-01-05,2026-06-18,2538,2420,871,582,967,0.215736,1.158006


In [6]:
# =========================
# 4. Merge Notebook02 strategy returns with lagged activity states
# =========================

base_activity_cols = [
    "date",
    "symbol",
    "activity_score",
    "activity_pctile",
    "activity_bucket",
    "volume_z",
    "open_interest_z",
    "turnover_value_z",
]

# Execution-date activity is kept only for diagnostics.
execution_activity = activity_features[base_activity_cols].rename(
    columns={
        "activity_score": "execution_activity_score",
        "activity_pctile": "execution_activity_pctile",
        "activity_bucket": "execution_activity_bucket",
        "volume_z": "execution_volume_z",
        "open_interest_z": "execution_open_interest_z",
        "turnover_value_z": "execution_turnover_value_z",
    }
)

strategy_activity = strategy.merge(
    execution_activity,
    on=["symbol", "date"],
    how="left",
    validate="many_to_one",
)

# Main activity state is lagged to signal_date, matching 02 delayed signal timing.
signal_activity = activity_features[base_activity_cols].rename(
    columns={
        "date": "signal_date",
        "activity_score": "signal_activity_score",
        "activity_pctile": "signal_activity_pctile",
        "activity_bucket": "signal_activity_bucket",
        "volume_z": "signal_volume_z",
        "open_interest_z": "signal_open_interest_z",
        "turnover_value_z": "signal_turnover_value_z",
    }
)

strategy_activity = strategy_activity.merge(
    signal_activity,
    on=["symbol", "signal_date"],
    how="left",
    validate="many_to_one",
)

# Standard analysis names refer to lagged signal-date activity state.
strategy_activity["activity_state_date"] = strategy_activity["signal_date"]
strategy_activity["activity_score"] = strategy_activity["signal_activity_score"]
strategy_activity["activity_pctile"] = strategy_activity["signal_activity_pctile"]
strategy_activity["activity_bucket"] = strategy_activity["signal_activity_bucket"]
strategy_activity["volume_z"] = strategy_activity["signal_volume_z"]
strategy_activity["open_interest_z"] = strategy_activity["signal_open_interest_z"]
strategy_activity["turnover_value_z"] = strategy_activity["signal_turnover_value_z"]

strategy_activity["activity_bucket"] = pd.Categorical(
    strategy_activity["activity_bucket"],
    categories=ACTIVITY_BUCKET_ORDER,
    ordered=True,
)

strategy_activity["execution_activity_bucket"] = pd.Categorical(
    strategy_activity["execution_activity_bucket"],
    categories=ACTIVITY_BUCKET_ORDER,
    ordered=True,
)

strategy_activity["exec_vol_bucket"] = pd.Categorical(
    strategy_activity["exec_vol_bucket"],
    categories=VOL_BUCKET_ORDER,
    ordered=True,
)

analysis_panel = strategy_activity[
    strategy_activity["activity_bucket"].notna()
    & strategy_activity["exec_vol_bucket"].notna()
].copy()

print("strategy_activity shape:", strategy_activity.shape)
print("analysis_panel shape:", analysis_panel.shape)
print("dropped rows due to missing lagged activity/bucket:", len(strategy_activity) - len(analysis_panel))
display(analysis_panel.head())

strategy_activity shape: (452150, 45)
analysis_panel shape: (442754, 45)
dropped rows due to missing lagged activity/bucket: 9396


,date,signal_date,symbol,sector,close,fwd_ret_1d,fwd_log_ret_1d,fwd_tradable_return_flag,vol_bucket,rv_60,ma_value,signal_close,ma_window,version,position,valid_strategy_row,turnover,cost_return,passive_return,gross_strategy_return,net_strategy_return,gross_map_vs_passive,net_map_vs_passive,net_map_vs_zero,exec_position,exec_vol_bucket,execution_activity_score,execution_activity_pctile,execution_activity_bucket,execution_volume_z,execution_open_interest_z,execution_turnover_value_z,signal_activity_score,signal_activity_pctile,signal_activity_bucket,signal_volume_z,signal_open_interest_z,signal_turnover_value_z,activity_state_date,activity_score,activity_pctile,activity_bucket,volume_z,open_interest_z,turnover_value_z
58,2016-07-01,2016-06-30,AG,precious_metal,4230.0,0.044681,0.043711,True,Mid,0.015633,3900.2,4064.0,10,long_flat,1.0,True,0.0,0.0,0.044681,0.044681,0.044681,0.0,0.0,0.044681,1.0,Mid,2.272562,0.975410,High,2.347554,1.852940,2.617192,1.495181,0.908333,High,1.038760,2.149630,1.297151,2016-06-30,1.495181,0.908333,High,1.038760,2.149630,1.297151
59,2016-07-04,2016-07-01,AG,precious_metal,4419.0,0.001810,0.001809,True,Mid,0.016198,3939.6,4230.0,10,long_flat,1.0,True,0.0,0.0,0.001810,0.001810,0.001810,0.0,0.0,0.001810,1.0,Mid,1.420265,0.862903,High,1.057860,1.738116,1.464819,2.272562,0.975410,High,2.347554,1.852940,2.617192,2016-07-01,2.272562,0.975410,High,2.347554,1.852940,2.617192
60,2016-07-05,2016-07-04,AG,precious_metal,4427.0,0.018974,0.018797,True,Mid,0.017022,3997.4,4419.0,10,long_flat,1.0,True,0.0,0.0,0.018974,0.018974,0.018974,0.0,0.0,0.018974,1.0,Mid,2.332482,0.976190,High,2.747533,1.211206,3.038706,1.420265,0.862903,High,1.057860,1.738116,1.464819,2016-07-04,1.420265,0.862903,High,1.057860,1.738116,1.464819
61,2016-07-06,2016-07-05,AG,precious_metal,4511.0,-0.012414,-0.012492,True,Mid,0.017022,4055.1,4427.0,10,long_flat,1.0,True,0.0,0.0,-0.012414,-0.012414,-0.012414,0.0,0.0,-0.012414,1.0,Mid,2.421503,0.992188,High,2.822082,1.336481,3.105945,2.332482,0.976190,High,2.747533,1.211206,3.038706,2016-07-05,2.332482,0.976190,High,2.747533,1.211206,3.038706
62,2016-07-07,2016-07-06,AG,precious_metal,4455.0,-0.025814,-0.026153,True,Mid,0.017114,4125.3,4511.0,10,long_flat,1.0,True,0.0,0.0,-0.025814,-0.025814,-0.025814,0.0,0.0,-0.025814,1.0,Mid,1.576617,0.884615,High,1.582156,1.244816,1.902878,2.421503,0.992188,High,2.822082,1.336481,3.105945,2016-07-06,2.421503,0.992188,High,2.822082,1.336481,3.105945


In [7]:
# =========================
# 5. Daily portfolio returns by lagged activity bucket and vol bucket
# =========================

activity_bucket_daily_returns = (
    analysis_panel
    .groupby(["date", "ma_window", "version", "activity_bucket"], observed=True)[available_return_cols]
    .mean()
    .reset_index()
)

activity_vol_bucket_daily_returns = (
    analysis_panel
    .groupby(["date", "ma_window", "version", "activity_bucket", "exec_vol_bucket"], observed=True)[available_return_cols]
    .mean()
    .reset_index()
)

activity_bucket_counts = (
    analysis_panel
    .groupby(["date", "ma_window", "version", "activity_bucket", "exec_vol_bucket"], observed=True)
    .agg(n_symbols=("symbol", "nunique"), n_rows=("symbol", "size"))
    .reset_index()
)

print("activity_bucket_daily_returns shape:", activity_bucket_daily_returns.shape)
print("activity_vol_bucket_daily_returns shape:", activity_vol_bucket_daily_returns.shape)
display(activity_vol_bucket_daily_returns.head())

activity_bucket_daily_returns shape: (57722, 10)
activity_vol_bucket_daily_returns shape: (157846, 11)


,date,ma_window,version,activity_bucket,exec_vol_bucket,gross_strategy_return,net_strategy_return,passive_return,gross_map_vs_passive,net_map_vs_passive,net_map_vs_zero
0,2016-07-01,10,long_flat,Low,Low,0.012822,0.012822,0.012822,0.0,0.0000,0.012822
1,2016-07-01,10,long_flat,Low,Mid,0.020737,0.020737,0.020737,0.0,0.0000,0.020737
2,2016-07-01,10,long_flat,Low,High,0.016039,0.016039,0.016039,0.0,0.0000,0.016039
3,2016-07-01,10,long_flat,Mid,Low,0.040424,0.039924,0.040424,0.0,-0.0005,0.039924
4,2016-07-01,10,long_flat,Mid,Mid,0.021440,0.021440,0.021440,0.0,0.0000,0.021440


In [8]:
# =========================
# 6. High-Vol-minus-Low-Vol returns within each lagged activity state
# =========================

hml_idx = ["date", "ma_window", "version", "activity_bucket"]

hml_frames = []
for rc in available_return_cols:
    p = activity_vol_bucket_daily_returns.pivot_table(
        index=hml_idx,
        columns="exec_vol_bucket",
        values=rc,
        aggfunc="mean",
        observed=True,
    )

    if ("High" not in p.columns) or ("Low" not in p.columns):
        continue

    tmp = p[["High", "Low"]].copy()
    tmp[rc] = tmp["High"] - tmp["Low"]
    tmp = tmp[[rc]].reset_index()
    hml_frames.append(tmp)

if not hml_frames:
    raise ValueError("No High-Low HML frames were created. Check activity/vol bucket coverage.")

hml_activity_daily_returns = hml_frames[0]
for tmp in hml_frames[1:]:
    hml_activity_daily_returns = hml_activity_daily_returns.merge(tmp, on=hml_idx, how="outer")

hml_activity_daily_returns = hml_activity_daily_returns.dropna(subset=available_return_cols, how="all")
hml_activity_daily_returns["activity_bucket"] = pd.Categorical(
    hml_activity_daily_returns["activity_bucket"],
    categories=ACTIVITY_BUCKET_ORDER,
    ordered=True,
)

print("hml_activity_daily_returns shape:", hml_activity_daily_returns.shape)
display(hml_activity_daily_returns.head())

hml_activity_daily_returns shape: (48018, 10)


exec_vol_bucket,date,ma_window,version,activity_bucket,gross_strategy_return,net_strategy_return,passive_return,gross_map_vs_passive,net_map_vs_passive,net_map_vs_zero
0,2016-07-01,10,long_flat,Low,0.003217,0.003217,0.003217,0.0,0.0000,0.003217
1,2016-07-01,10,long_flat,Mid,-0.004013,-0.003513,-0.004013,0.0,0.0005,-0.003513
2,2016-07-01,10,long_flat,High,0.026999,0.026999,0.026999,0.0,0.0000,0.026999
3,2016-07-01,10,long_short,Low,0.003217,0.003217,0.003217,0.0,0.0000,0.003217
4,2016-07-01,10,long_short,Mid,-0.004013,-0.003013,-0.004013,0.0,0.0010,-0.003013


In [9]:
# =========================
# 7. Activity attenuation series:
#    High-Activity HML minus Low-Activity HML
#    Negative value = MA High-Vol advantage is weaker in high-activity states.
# =========================

atten_idx = ["date", "ma_window", "version"]

attenuation_frames = []
for rc in available_return_cols:
    p = hml_activity_daily_returns.pivot_table(
        index=atten_idx,
        columns="activity_bucket",
        values=rc,
        aggfunc="mean",
        observed=True,
    )

    if ("High" not in p.columns) or ("Low" not in p.columns):
        continue

    tmp = p[["High", "Low"]].copy()
    tmp[rc] = tmp["High"] - tmp["Low"]
    tmp = tmp[[rc]].reset_index()
    attenuation_frames.append(tmp)

if not attenuation_frames:
    raise ValueError("No activity attenuation frames were created. Check High/Low activity coverage.")

activity_attenuation_daily_returns = attenuation_frames[0]
for tmp in attenuation_frames[1:]:
    activity_attenuation_daily_returns = activity_attenuation_daily_returns.merge(tmp, on=atten_idx, how="outer")

activity_attenuation_daily_returns = activity_attenuation_daily_returns.dropna(subset=available_return_cols, how="all")

print("activity_attenuation_daily_returns shape:", activity_attenuation_daily_returns.shape)
display(activity_attenuation_daily_returns.head())

activity_attenuation_daily_returns shape: (12430, 9)


activity_bucket,date,ma_window,version,gross_strategy_return,net_strategy_return,passive_return,gross_map_vs_passive,net_map_vs_passive,net_map_vs_zero
0,2016-07-01,10,long_flat,0.023782,0.023782,0.023782,0.0,0.000000,0.023782
1,2016-07-01,10,long_short,0.023782,0.023782,0.023782,0.0,0.000000,0.023782
2,2016-07-01,20,long_flat,0.023782,0.023782,0.023782,0.0,0.000000,0.023782
3,2016-07-01,20,long_short,0.023782,0.023782,0.023782,0.0,0.000000,0.023782
4,2016-07-01,60,long_flat,0.023782,0.023616,0.023782,0.0,-0.000167,0.023616


In [10]:
# =========================
# 8. Summary tables
# =========================

activity_bucket_performance_summary = summarize_returns(
    activity_bucket_daily_returns,
    ["ma_window", "version", "activity_bucket"],
    available_return_cols,
)

activity_vol_bucket_performance_summary = summarize_returns(
    activity_vol_bucket_daily_returns,
    ["ma_window", "version", "activity_bucket", "exec_vol_bucket"],
    available_return_cols,
)

hml_activity_summary = summarize_returns(
    hml_activity_daily_returns,
    ["ma_window", "version", "activity_bucket"],
    available_return_cols,
)

attenuation_summary = summarize_returns(
    activity_attenuation_daily_returns,
    ["ma_window", "version"],
    available_return_cols,
).rename(
    columns={
        "mean_daily": "high_minus_low_activity_hml_mean_daily",
        "mean_ann": "high_minus_low_activity_hml_mean_ann",
        "sharpe": "high_minus_low_activity_hml_sharpe",
        "t_stat": "high_minus_low_activity_hml_t_stat",
        "vol_daily": "high_minus_low_activity_hml_vol_daily",
        "vol_ann": "high_minus_low_activity_hml_vol_ann",
        "max_drawdown": "high_minus_low_activity_hml_max_drawdown",
    }
)

hml_mean_pivot = (
    hml_activity_summary
    .pivot_table(
        index=["ma_window", "version", "return_col"],
        columns="activity_bucket",
        values="mean_ann",
        aggfunc="first",
        observed=True,
    )
    .reset_index()
    .rename(
        columns={
            "Low": "low_activity_hml_mean_ann",
            "Mid": "mid_activity_hml_mean_ann",
            "High": "high_activity_hml_mean_ann",
        }
    )
)

attenuation_summary = attenuation_summary.merge(
    hml_mean_pivot,
    on=["ma_window", "version", "return_col"],
    how="left",
)

attenuation_summary["activity_effect_flag"] = attenuation_summary.apply(
    lambda r: activity_effect_flag(
        r["high_minus_low_activity_hml_t_stat"],
        r["high_minus_low_activity_hml_mean_ann"],
        int(r["n_obs"]) if pd.notna(r["n_obs"]) else 0,
    ),
    axis=1,
)

print("activity_bucket_performance_summary:", activity_bucket_performance_summary.shape)
print("activity_vol_bucket_performance_summary:", activity_vol_bucket_performance_summary.shape)
print("hml_activity_summary:", hml_activity_summary.shape)
print("attenuation_summary:", attenuation_summary.shape)

display(
    attenuation_summary[
        attenuation_summary["return_col"].isin(available_core_return_cols)
    ].sort_values(["ma_window", "version", "return_col"]).head(20)
)

activity_bucket_performance_summary: (144, 14)
activity_vol_bucket_performance_summary: (432, 15)
hml_activity_summary: (144, 14)
attenuation_summary: (48, 17)


,ma_window,version,return_col,n_obs,high_minus_low_activity_hml_mean_daily,high_minus_low_activity_hml_mean_ann,high_minus_low_activity_hml_vol_daily,high_minus_low_activity_hml_vol_ann,high_minus_low_activity_hml_sharpe,high_minus_low_activity_hml_t_stat,min_daily,max_daily,high_minus_low_activity_hml_max_drawdown,high_activity_hml_mean_ann,low_activity_hml_mean_ann,mid_activity_hml_mean_ann,activity_effect_flag
4,10,long_flat,net_map_vs_passive,1554,0.000098,0.024819,0.015780,0.250493,0.099079,0.246042,-0.099942,0.111373,-0.677738,-0.002449,0.017152,-0.055005,WEAK_ENHANCEMENT
5,10,long_flat,net_map_vs_zero,1554,0.000040,0.010118,0.017314,0.274845,0.036814,0.091418,-0.079927,0.100884,-0.595631,0.044275,0.012477,-0.031843,WEAK_ENHANCEMENT
1,10,long_flat,net_strategy_return,1554,0.000040,0.010118,0.017314,0.274845,0.036814,0.091418,-0.079927,0.100884,-0.595631,0.044275,0.012477,-0.031843,WEAK_ENHANCEMENT
10,10,long_short,net_map_vs_passive,1554,0.000197,0.049637,0.031559,0.500986,0.099079,0.246042,-0.199883,0.222747,-0.916011,-0.004898,0.034303,-0.110009,WEAK_ENHANCEMENT
11,10,long_short,net_map_vs_zero,1554,0.000139,0.034937,0.024504,0.388992,0.089813,0.223032,-0.130329,0.146890,-0.792191,0.041826,0.029628,-0.086848,WEAK_ENHANCEMENT
7,10,long_short,net_strategy_return,1554,0.000139,0.034937,0.024504,0.388992,0.089813,0.223032,-0.130329,0.146890,-0.792191,0.041826,0.029628,-0.086848,WEAK_ENHANCEMENT
16,20,long_flat,net_map_vs_passive,1554,-0.000042,-0.010554,0.015337,0.243473,-0.043346,-0.107641,-0.100732,0.092844,-0.703311,0.008415,0.058666,-0.005045,WEAK_ATTENUATION
17,20,long_flat,net_map_vs_zero,1554,-0.000100,-0.025254,0.017599,0.279382,-0.090394,-0.224472,-0.079927,0.100884,-0.738079,0.055138,0.053991,0.018117,WEAK_ATTENUATION
13,20,long_flat,net_strategy_return,1554,-0.000100,-0.025254,0.017599,0.279382,-0.090394,-0.224472,-0.079927,0.100884,-0.738079,0.055138,0.053991,0.018117,WEAK_ATTENUATION
22,20,long_short,net_map_vs_passive,1554,-0.000084,-0.021107,0.030675,0.486945,-0.043346,-0.107641,-0.201465,0.185687,-0.927681,0.016830,0.117332,-0.010089,WEAK_ATTENUATION


In [11]:
# =========================
# 9. Interaction regression:
#    return ~ high_vol + lagged_activity_score + high_vol * lagged_activity_score
#
# Interpretation:
# - interaction < 0 means the High-Vol advantage weakens when prior activity rises.
# - Use HC1 robust standard errors.
# =========================

regression_rows = []

reg_base = analysis_panel[
    analysis_panel["exec_vol_bucket"].isin(["Low", "High"])
    & analysis_panel["activity_score"].notna()
].copy()

reg_base["high_vol_dummy"] = (reg_base["exec_vol_bucket"].astype(str) == "High").astype(float)
reg_base["activity_score_reg"] = pd.to_numeric(reg_base["activity_score"], errors="coerce").clip(-5, 5)
reg_base["high_vol_x_activity_score"] = (
    reg_base["high_vol_dummy"] * reg_base["activity_score_reg"]
)

reg_terms = [
    "const",
    "high_vol_dummy",
    "activity_score",
    "high_vol_x_activity_score",
]

for (ma_window, version), g in reg_base.groupby(["ma_window", "version"]):
    X = np.column_stack(
        [
            np.ones(len(g)),
            g["high_vol_dummy"].to_numpy(float),
            g["activity_score_reg"].to_numpy(float),
            g["high_vol_x_activity_score"].to_numpy(float),
        ]
    )

    for rc in available_core_return_cols:
        y = pd.to_numeric(g[rc], errors="coerce").to_numpy(float)
        res = ols_hc1(y, X, reg_terms)
        res["ma_window"] = ma_window
        res["version"] = version
        res["return_col"] = rc
        regression_rows.append(res)

if not regression_rows:
    raise ValueError("No interaction regression rows were created. Check Low/High vol coverage and activity_score availability.")

activity_interaction_regression = pd.concat(regression_rows, ignore_index=True)

interaction_flags = (
    activity_interaction_regression[
        activity_interaction_regression["term"] == "high_vol_x_activity_score"
    ]
    .copy()
    .rename(
        columns={
            "coef": "interaction_coef",
            "std_err_hc1": "interaction_std_err_hc1",
            "t_stat_hc1": "interaction_t_stat_hc1",
        }
    )
)

interaction_flags["regression_activity_effect_flag"] = interaction_flags.apply(
    lambda r: regression_effect_flag(
        r["interaction_t_stat_hc1"],
        r["interaction_coef"],
        int(r["n_obs"]) if pd.notna(r["n_obs"]) else 0,
    ),
    axis=1,
)

interaction_flags = interaction_flags[
    [
        "ma_window",
        "version",
        "return_col",
        "interaction_coef",
        "interaction_std_err_hc1",
        "interaction_t_stat_hc1",
        "n_obs",
        "r2",
        "regression_activity_effect_flag",
    ]
].rename(
    columns={
        "n_obs": "regression_n_obs",
        "r2": "regression_r2",
    }
).sort_values(["ma_window", "version", "return_col"]).reset_index(drop=True)

display(interaction_flags)


,ma_window,version,return_col,interaction_coef,interaction_std_err_hc1,interaction_t_stat_hc1,regression_n_obs,regression_r2,regression_activity_effect_flag
0,10,long_flat,net_map_vs_passive,0.000131,0.000146,0.891692,36697,0.000052,WEAK_ENHANCEMENT
1,10,long_flat,net_map_vs_zero,0.000141,0.000171,0.824102,36697,0.000152,WEAK_ENHANCEMENT
2,10,long_flat,net_strategy_return,0.000141,0.000171,0.824102,36697,0.000152,WEAK_ENHANCEMENT
3,10,long_short,net_map_vs_passive,0.000261,0.000293,0.891692,36697,0.000052,WEAK_ENHANCEMENT
4,10,long_short,net_map_vs_zero,0.000271,0.000225,1.205844,36697,0.000120,WEAK_ENHANCEMENT
5,10,long_short,net_strategy_return,0.000271,0.000225,1.205844,36697,0.000120,WEAK_ENHANCEMENT
6,20,long_flat,net_map_vs_passive,-0.000035,0.000148,-0.237893,36697,0.000021,WEAK_ATTENUATION
7,20,long_flat,net_map_vs_zero,-0.000025,0.000169,-0.147447,36697,0.000100,WEAK_ATTENUATION
8,20,long_flat,net_strategy_return,-0.000025,0.000169,-0.147447,36697,0.000100,WEAK_ATTENUATION
9,20,long_short,net_map_vs_passive,-0.000070,0.000296,-0.237893,36697,0.000021,WEAK_ATTENUATION


In [12]:
# =========================
# 10. Final activity flags
# =========================

activity_flags = attenuation_summary[
    attenuation_summary["return_col"].isin(available_core_return_cols)
][
    [
        "ma_window",
        "version",
        "return_col",
        "low_activity_hml_mean_ann",
        "mid_activity_hml_mean_ann",
        "high_activity_hml_mean_ann",
        "high_minus_low_activity_hml_mean_ann",
        "high_minus_low_activity_hml_t_stat",
        "high_minus_low_activity_hml_sharpe",
        "n_obs",
        "activity_effect_flag",
    ]
].rename(
    columns={"n_obs": "attenuation_n_obs"}
).copy()

activity_flags = activity_flags.merge(
    interaction_flags,
    on=["ma_window", "version", "return_col"],
    how="left",
)


def combined_flag(row):
    bucket_flag = row["activity_effect_flag"]
    reg_flag = row.get("regression_activity_effect_flag", None)

    if bucket_flag == "ATTENUATED" or reg_flag == "ATTENUATED":
        return "ATTENUATED"
    if bucket_flag == "ENHANCED_BY_ACTIVITY" or reg_flag == "ENHANCED_BY_ACTIVITY":
        return "ENHANCED_BY_ACTIVITY"
    if bucket_flag == "WEAK_ATTENUATION" and reg_flag == "WEAK_ATTENUATION":
        return "WEAK_ATTENUATION"
    if bucket_flag == "WEAK_ENHANCEMENT" and reg_flag == "WEAK_ENHANCEMENT":
        return "WEAK_ENHANCEMENT"
    if "INSUFFICIENT" in str(bucket_flag):
        return "INSUFFICIENT_SAMPLE"
    return "MIXED_OR_WEAK"


activity_flags["combined_activity_effect_flag"] = activity_flags.apply(combined_flag, axis=1)
activity_flags = activity_flags.sort_values(["ma_window", "version", "return_col"]).reset_index(drop=True)

display(activity_flags)

,ma_window,version,return_col,low_activity_hml_mean_ann,mid_activity_hml_mean_ann,high_activity_hml_mean_ann,high_minus_low_activity_hml_mean_ann,high_minus_low_activity_hml_t_stat,high_minus_low_activity_hml_sharpe,attenuation_n_obs,activity_effect_flag,interaction_coef,interaction_std_err_hc1,interaction_t_stat_hc1,regression_n_obs,regression_r2,regression_activity_effect_flag,combined_activity_effect_flag
0,10,long_flat,net_map_vs_passive,0.017152,-0.055005,-0.002449,0.024819,0.246042,0.099079,1554,WEAK_ENHANCEMENT,0.000131,0.000146,0.891692,36697,0.000052,WEAK_ENHANCEMENT,WEAK_ENHANCEMENT
1,10,long_flat,net_map_vs_zero,0.012477,-0.031843,0.044275,0.010118,0.091418,0.036814,1554,WEAK_ENHANCEMENT,0.000141,0.000171,0.824102,36697,0.000152,WEAK_ENHANCEMENT,WEAK_ENHANCEMENT
2,10,long_flat,net_strategy_return,0.012477,-0.031843,0.044275,0.010118,0.091418,0.036814,1554,WEAK_ENHANCEMENT,0.000141,0.000171,0.824102,36697,0.000152,WEAK_ENHANCEMENT,WEAK_ENHANCEMENT
3,10,long_short,net_map_vs_passive,0.034303,-0.110009,-0.004898,0.049637,0.246042,0.099079,1554,WEAK_ENHANCEMENT,0.000261,0.000293,0.891692,36697,0.000052,WEAK_ENHANCEMENT,WEAK_ENHANCEMENT
4,10,long_short,net_map_vs_zero,0.029628,-0.086848,0.041826,0.034937,0.223032,0.089813,1554,WEAK_ENHANCEMENT,0.000271,0.000225,1.205844,36697,0.000120,WEAK_ENHANCEMENT,WEAK_ENHANCEMENT
5,10,long_short,net_strategy_return,0.029628,-0.086848,0.041826,0.034937,0.223032,0.089813,1554,WEAK_ENHANCEMENT,0.000271,0.000225,1.205844,36697,0.000120,WEAK_ENHANCEMENT,WEAK_ENHANCEMENT
6,20,long_flat,net_map_vs_passive,0.058666,-0.005045,0.008415,-0.010554,-0.107641,-0.043346,1554,WEAK_ATTENUATION,-0.000035,0.000148,-0.237893,36697,0.000021,WEAK_ATTENUATION,WEAK_ATTENUATION
7,20,long_flat,net_map_vs_zero,0.053991,0.018117,0.055138,-0.025254,-0.224472,-0.090394,1554,WEAK_ATTENUATION,-0.000025,0.000169,-0.147447,36697,0.000100,WEAK_ATTENUATION,WEAK_ATTENUATION
8,20,long_flat,net_strategy_return,0.053991,0.018117,0.055138,-0.025254,-0.224472,-0.090394,1554,WEAK_ATTENUATION,-0.000025,0.000169,-0.147447,36697,0.000100,WEAK_ATTENUATION,WEAK_ATTENUATION
9,20,long_short,net_map_vs_passive,0.117332,-0.010089,0.016830,-0.021107,-0.107641,-0.043346,1554,WEAK_ATTENUATION,-0.000070,0.000296,-0.237893,36697,0.000021,WEAK_ATTENUATION,WEAK_ATTENUATION


In [13]:
# =========================
# 11. Hard self-checks, including no-lookahead checks
# =========================

errors = []


def check(condition, message):
    if not bool(condition):
        errors.append(message)


# Input structure
check(clean["symbol"].nunique() == 27, "Clean panel should contain 27 symbols.")
check(strategy["symbol"].nunique() == 27, "Notebook02 strategy panel should contain 27 symbols.")
check(not strategy.duplicated(["date", "symbol", "ma_window", "version"]).any(),
      "Notebook02 strategy panel has duplicate date-symbol-ma-version rows.")

# 02 delayed timing
check((strategy["signal_date"] < strategy["date"]).all(),
      "Notebook02 delayed strategy timing failed: signal_date must be earlier than date.")

# Activity feature uniqueness and values
check(not activity_features.duplicated(["date", "symbol"]).any(),
      "Activity features have duplicate symbol-date rows.")

valid_pct = activity_features["activity_pctile"].dropna()
check(((valid_pct >= 0) & (valid_pct <= 1)).all(),
      "activity_pctile must be in [0, 1].")

valid_buckets = set(activity_features["activity_bucket"].dropna().unique())
check(valid_buckets.issubset(set(ACTIVITY_BUCKET_ORDER)),
      f"Unexpected activity bucket labels: {valid_buckets}")

# Main no-lookahead checks: analysis activity must be signal-date activity, not execution-date activity.
check(not analysis_panel.empty, "analysis_panel is empty.")
check((analysis_panel["activity_state_date"] == analysis_panel["signal_date"]).all(),
      "activity_state_date must equal signal_date.")
check((analysis_panel["activity_state_date"] < analysis_panel["date"]).all(),
      "Main activity state must be strictly earlier than execution date.")
check((analysis_panel["signal_activity_bucket"].astype(str) == analysis_panel["activity_bucket"].astype(str)).all(),
      "Main activity_bucket must equal signal_activity_bucket.")
check(analysis_panel["activity_bucket"].notna().all(), "analysis_panel contains missing lagged activity_bucket.")
check(analysis_panel["exec_vol_bucket"].notna().all(), "analysis_panel contains missing exec_vol_bucket.")

# Merge correctness: lagged activity score must equal activity_features at (symbol, signal_date).
activity_lookup = activity_features[["symbol", "date", "activity_score", "activity_bucket"]].rename(
    columns={"date": "signal_date", "activity_score": "expected_activity_score", "activity_bucket": "expected_activity_bucket"}
)
_check_merge = analysis_panel[["symbol", "signal_date", "activity_score", "activity_bucket"]].merge(
    activity_lookup,
    on=["symbol", "signal_date"],
    how="left",
    validate="many_to_one",
)
score_diff = (_check_merge["activity_score"] - _check_merge["expected_activity_score"]).abs().max(skipna=True)
check(pd.isna(score_diff) or score_diff < 1e-12, "Lagged activity_score merge mismatch.")
check((_check_merge["activity_bucket"].astype(str) == _check_merge["expected_activity_bucket"].astype(str)).all(),
      "Lagged activity_bucket merge mismatch.")

for rc in available_return_cols:
    check(analysis_panel[rc].notna().all(), f"analysis_panel has NaN in {rc}.")

# Daily portfolio keys
check(not activity_bucket_daily_returns.duplicated(["date", "ma_window", "version", "activity_bucket"]).any(),
      "Duplicate activity-bucket daily portfolio rows.")

check(not activity_vol_bucket_daily_returns.duplicated(
    ["date", "ma_window", "version", "activity_bucket", "exec_vol_bucket"]
).any(), "Duplicate activity-vol-bucket daily portfolio rows.")

check(not hml_activity_daily_returns.duplicated(["date", "ma_window", "version", "activity_bucket"]).any(),
      "Duplicate HML activity daily rows.")

check(not activity_attenuation_daily_returns.duplicated(["date", "ma_window", "version"]).any(),
      "Duplicate activity attenuation daily rows.")

# Summary non-empty
for name, df in {
    "activity_bucket_performance_summary": activity_bucket_performance_summary,
    "activity_vol_bucket_performance_summary": activity_vol_bucket_performance_summary,
    "hml_activity_summary": hml_activity_summary,
    "attenuation_summary": attenuation_summary,
    "activity_interaction_regression": activity_interaction_regression,
    "activity_flags": activity_flags,
}.items():
    check(not df.empty, f"{name} is empty.")

core_flags = activity_flags[activity_flags["return_col"].isin(available_core_return_cols)]
check(not core_flags.empty, "Core activity flags are empty.")

if errors:
    print("Notebook 03 self-check FAILED:")
    for e in errors:
        print(" -", e)
    raise AssertionError("Notebook 03 hard self-check failed.")
else:
    print("Notebook 03 hard self-check: PASSED")

Notebook 03 hard self-check: PASSED


In [14]:
# =========================
# 12. Save outputs
# =========================

saved = []

saved.append(save_table(activity_features, FEATURE_DIR, "notebook03_activity_features"))
saved.append(save_table(activity_feature_summary, TABLE_DIR, "notebook03_activity_feature_summary"))
saved.append(save_table(activity_bucket_counts, DIAG_DIR, "notebook03_activity_bucket_counts"))

saved.append(save_table(strategy_activity, STRATEGY_DIR, "notebook03_strategy_activity_panel"))
saved.append(save_table(analysis_panel, STRATEGY_DIR, "notebook03_strategy_activity_analysis_panel"))

saved.append(save_table(activity_bucket_daily_returns, STRATEGY_DIR, "notebook03_activity_bucket_daily_returns"))
saved.append(save_table(activity_vol_bucket_daily_returns, STRATEGY_DIR, "notebook03_activity_vol_bucket_daily_returns"))
saved.append(save_table(hml_activity_daily_returns, STRATEGY_DIR, "notebook03_hml_by_activity_daily_returns"))
saved.append(save_table(activity_attenuation_daily_returns, STRATEGY_DIR, "notebook03_activity_attenuation_daily_returns"))

saved.append(save_table(activity_bucket_performance_summary, TABLE_DIR, "notebook03_activity_bucket_performance_summary"))
saved.append(save_table(activity_vol_bucket_performance_summary, TABLE_DIR, "notebook03_activity_vol_bucket_performance_summary"))
saved.append(save_table(hml_activity_summary, TABLE_DIR, "notebook03_hml_by_activity_summary"))
saved.append(save_table(attenuation_summary, TABLE_DIR, "notebook03_activity_attenuation_summary"))
saved.append(save_table(activity_interaction_regression, TABLE_DIR, "notebook03_activity_interaction_regression"))
saved.append(save_table(interaction_flags, TABLE_DIR, "notebook03_interaction_flags"))
saved.append(save_table(activity_flags, TABLE_DIR, "notebook03_activity_flags"))

run_config_path = NB03_DIR / "notebook03_run_config.json"
with open(run_config_path, "w", encoding="utf-8") as f:
    json.dump(RUN_CONFIG, f, indent=2, ensure_ascii=False)

saved_summary = pd.DataFrame(saved)
saved_summary.to_csv(NB03_DIR / "notebook03_saved_outputs.csv", index=False)

display(saved_summary)

print("Notebook 03 finished.")
print("Main result table:")
print(TABLE_DIR / "notebook03_activity_flags.csv")
print("Main detailed attenuation table:")
print(TABLE_DIR / "notebook03_activity_attenuation_summary.csv")
print("Main strategy-activity panel for later notebooks:")
print(STRATEGY_DIR / "notebook03_strategy_activity_analysis_panel.csv")

,name,rows,cols,csv,parquet
0,notebook03_activity_features,58591,8,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
1,notebook03_activity_feature_summary,27,10,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
2,notebook03_activity_bucket_counts,157846,7,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
3,notebook03_strategy_activity_panel,452150,45,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
4,notebook03_strategy_activity_analysis_panel,442754,45,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
5,notebook03_activity_bucket_daily_returns,57722,10,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
6,notebook03_activity_vol_bucket_daily_returns,157846,11,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
7,notebook03_hml_by_activity_daily_returns,48018,10,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
8,notebook03_activity_attenuation_daily_returns,12430,9,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...
9,notebook03_activity_bucket_performance_summary,144,14,/home/zilinm2/futures_anomaly_project/notebook...,/home/zilinm2/futures_anomaly_project/notebook...


Notebook 03 finished.
Main result table:
/home/zilinm2/futures_anomaly_project/notebook03_liquidity_attenuation/tables/notebook03_activity_flags.csv
Main detailed attenuation table:
/home/zilinm2/futures_anomaly_project/notebook03_liquidity_attenuation/tables/notebook03_activity_attenuation_summary.csv
Main strategy-activity panel for later notebooks:
/home/zilinm2/futures_anomaly_project/notebook03_liquidity_attenuation/strategy_returns/notebook03_strategy_activity_analysis_panel.csv
